In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

import os
print(os.listdir(path))

In [ ]:
# Write your code here
import os
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

# Custom Dataset
class PotatoDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_labels = {cls_name: i for i, cls_name in enumerate(classes)}
        self.image_paths = []
        self.labels = []
        for class_name, label in self.class_labels.items():
            paths = glob.glob(f"{root_dir}/{class_name}/*.*") # jpg, png...
            self.image_paths.extend(paths)
            self.labels.extend([label]*len(paths))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        return image, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])
test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])
train_dir = "/kaggle/input/q1-stage-3-2026/PlantVillage/train"
test_dir  = "/kaggle/input/q1-stage-3-2026/PlantVillage/test"


# Datasets
train_dataset = PotatoDataset(train_dir, transform=train_transform)
test_dataset = PotatoDataset(test_dir, transform=test_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Display some sample images
def imshow(img, label):
    plt.imshow(img.permute(1,2,0))
    plt.title(label)
    plt.axis('off')
    plt.show()

dataiter = iter(train_loader)
images, labels = next(dataiter)
for i in range(4):
    imshow(images[i], list(train_dataset.class_labels.keys())[labels[i]])

In [ ]:
# Write your code here
import torch
import torch.nn as nn

class PotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNN, self).__init__()
        # 5 Conv layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1) #conv1
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)#conv2
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)#conv3
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)#conv4
        self.bn4 = nn.BatchNorm2d(256)
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)#conv5
        self.bn5 = nn.BatchNorm2d(512)
        self.pool = nn.MaxPool2d(2,2)
        self.fc1 = nn.Linear(512*1*1, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.pool(self.relu(self.bn4(self.conv4(x))))
        x = self.pool(self.relu(self.bn5(self.conv5(x))))
        # Flatten
        x = x.view(x.size(0), -1)
        # Fully Connected Layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [ ]:
# Write your code here
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoCNN(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()# Reset gradients
        outputs = model(inputs)
        loss = criterion(outputs, labels)# compute LOSS
        loss.backward()
        optimizer.step()
        running_loss += loss.item()*inputs.size(0)
        _, preds = torch.max(outputs,1)
        total += labels.size(0)
        correct += (preds==labels).sum().item()
    return running_loss/total, correct/total

def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs) # Forward pass
            loss = criterion(outputs, labels)
            running_loss += loss.item()*inputs.size(0)
            _, preds = torch.max(outputs,1)
            total += labels.size(0)
            correct += (preds==labels).sum().item()
    return running_loss/total, correct/total

In [ ]:
# Write your code here
num_epochs = 10
train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_epoch(model, test_loader, criterion, device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    # Plotting losses and accuracy
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(train_losses,label="Train Loss")
plt.plot(val_losses,label="Val Loss")
plt.legend()
plt.subplot(1,2,2)
plt.plot(train_accs,label="Train Acc")
plt.plot(val_accs,label="Val Acc")
plt.legend()
plt.show()

In [ ]:
# Write your code here
